<a href="https://colab.research.google.com/github/aravindh28/swiftcounter-cv/blob/dev%2Fam2/Pipeline_v3_1_cleanup_with_validation_TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Setup

In [10]:

# -*- coding: utf-8 -*-
"""
Cleaned Bird Classification Pipeline
Optimized for multiple bird counting with simplified, efficient code
"""

# ============================================================================
# COLAB SETUP
# ============================================================================

# Install required packages
!pip install yt-dlp ultralytics opencv-python tqdm numpy

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# ============================================================================
# IMPORTS AND SETUP
# ============================================================================

import numpy as np
import cv2
import torch
import json
import math
from tqdm import tqdm
from ultralytics import YOLO, SAM
from datetime import datetime
import os


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Config


In [11]:


# ============================================================================
# CONFIGURATION
# ============================================================================

# Set your path - UPDATE THIS to match your Google Drive structure
PATH = "drive/MyDrive/UW-MS-DS/CSE493G1/Final_Project/Github/swiftcounter-cv/experiments"

# Check if path exists and create data directory if needed
!ls {PATH}
!mkdir -p {PATH}/data

print(f"✅ Working directory: {PATH}")
print(f"📁 Data directory: {PATH}/data")

# IMPORTANT: Ensure cookies.txt is in the data directory for YouTube downloads
print("📌 IMPORTANT: Make sure cookies.txt is in the data directory for video downloads!")
print(f"   Expected location: {PATH}/data/cookies.txt")

# ROI polygon (rectangle around chimney)
roi_polygon = np.array([
    [580, 550],  # top-left
    [780, 550],  # top-right
    [780, 720],  # bottom-right
    [580, 720],  # bottom-left
], dtype=int)

# Save ROI for use by tracker
roi_save_path = f"{PATH}/data/roi_polygon.npy"
np.save(roi_save_path, roi_polygon)
print(f"✅ ROI coordinates: {roi_polygon.tolist()}")
print(f"✅ ROI saved to: {roi_save_path}")

# Test segments configuration
segments_to_test = [
    {"start": "0:05", "end": "0:16", "expected_count": 47, "name": "segment_1_baseline"},
    {"start": "0:33", "end": "0:54", "expected_count": 71, "name": "segment_2_medium"},
    {"start": "0:54", "end": "1:14", "expected_count": 151, "name": "segment_3_high_traffic"},
    {"start": "1:40", "end": "1:50", "expected_count": 54, "name": "segment_4_short_burst"},
    {"start": "9:52", "end": "10:10", "expected_count": 145, "name": "segment_5_extreme_density"}
]


VIDEO_URL = "https://www.youtube.com/watch?v=mkquDCpPD9c"

data
✅ Working directory: drive/MyDrive/UW-MS-DS/CSE493G1/Final_Project/Github/swiftcounter-cv/experiments
📁 Data directory: drive/MyDrive/UW-MS-DS/CSE493G1/Final_Project/Github/swiftcounter-cv/experiments/data
📌 IMPORTANT: Make sure cookies.txt is in the data directory for video downloads!
   Expected location: drive/MyDrive/UW-MS-DS/CSE493G1/Final_Project/Github/swiftcounter-cv/experiments/data/cookies.txt
✅ ROI coordinates: [[580, 550], [780, 550], [780, 720], [580, 720]]
✅ ROI saved to: drive/MyDrive/UW-MS-DS/CSE493G1/Final_Project/Github/swiftcounter-cv/experiments/data/roi_polygon.npy


Helper functions


In [12]:
# ============================================================================
# HELPER FUNCTIONS (Including Integrated Video Download)
# ============================================================================

import subprocess
import yt_dlp

def time_str_to_seconds(time_str):
    """Convert MM:SS format to seconds for yt-dlp"""
    if ':' in time_str:
        parts = time_str.split(':')
        if len(parts) == 2:
            minutes, seconds = parts
            return int(minutes) * 60 + int(seconds)
        elif len(parts) == 3:
            hours, minutes, seconds = parts
            return int(hours) * 3600 + int(minutes) * 60 + int(seconds)
    return int(time_str)

def time_str_to_filename(time_str):
    """Convert MM:SS to filename-safe format"""
    return time_str.replace(':', '-')

def precise_trim(input_path, output_path, start, end):
    """Precise video trimming using ffmpeg for frame accuracy"""
    duration = end - start
    print(f"✂️  Trimming {input_path} from {start}s to {end}s into {output_path}")
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_path,
        "-ss", str(start),
        "-t", str(duration),
        "-c:v", "libx264",
        "-an",        # no audio
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode == 0:
        print("✅ Precise trim complete!")
        return True
    else:
        print(f"❌ ffmpeg trimming failed:\n{result.stderr.decode()}")
        return False

def download_video(url, filename, start=None, end=None):
    """Integrated video download using yt-dlp with optional precise trimming"""
    # Download to a temporary file if segmenting
    temp_filename = filename
    if start is not None and end is not None:
        temp_filename = filename.replace('.mp4', '_raw.mp4')

    if os.path.exists(filename) and os.path.getsize(filename) > 0:
        print(f"✅ File '{filename}' already exists and is non-empty. Skipping download.")
        return True

    ydl_opts = {
        "outtmpl": temp_filename,
        "format": "bestvideo[height<=720][ext=mp4]/bestvideo[height<=720]/best",
        "merge_output_format": "mp4",
        "quiet": False,
        "noplaylist": True,
    }

    # Use cookies if found in data/
    cookies_path = os.path.join(f"{PATH}/data", "cookies.txt")
    if os.path.exists(cookies_path):
        print(f"🔐 Using cookies from '{cookies_path}'")
        ydl_opts["cookies"] = cookies_path

    print(f"📥 Downloading video from {url} to {temp_filename}")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
            print("✅ Download complete!")
    except Exception as e:
        print(f"❌ Download failed:\n{e}")
        return False

    # If segment requested, trim with ffmpeg for frame accuracy
    if start is not None and end is not None:
        success = precise_trim(temp_filename, filename, start, end)
        if temp_filename != filename and os.path.exists(temp_filename):
            os.remove(temp_filename)  # Clean up temp file
        return success

    return True

def download_video_segment_unique(video_url, start_time, end_time, segment_name):
    """Download video segment with caching - only downloads if file doesn't exist"""
    start_sec = time_str_to_seconds(start_time)
    end_sec = time_str_to_seconds(end_time)

    # Create unique filename
    start_filename = time_str_to_filename(start_time)
    end_filename = time_str_to_filename(end_time)
    unique_filename = f"downloaded_video_{start_filename}_to_{end_filename}.mp4"
    unique_path = f"{PATH}/data/{unique_filename}"

    # CHECK IF FILE ALREADY EXISTS (CACHING FIX)
    if os.path.exists(unique_path) and os.path.getsize(unique_path) > 0:
        print(f"✅ Using existing file: {unique_filename}")
        return unique_path, unique_filename

    print(f"📥 Downloading segment '{segment_name}': {start_time} to {end_time}")
    print(f"   Time range: {start_sec}s to {end_sec}s")
    print(f"   Saving as: {unique_filename}")

    # Ensure the data directory exists
    os.makedirs(f"{PATH}/data", exist_ok=True)

    # Use integrated download function instead of external script
    success = download_video(video_url, unique_path, start_sec, end_sec)

    if success and os.path.exists(unique_path):
        print(f"✅ Video saved as: {unique_filename}")
        return unique_path, unique_filename
    else:
        print(f"❌ Failed to download video segment")
        return None, None

Bird tracker class


In [13]:
# ============================================================================
# SIMPLIFIED BIRD TRACKER CLASS WITH TIGHTENED VALIDATION
# ============================================================================

class AggressiveBirdTracker:
    """Simplified bird tracker focused on multiple bird detection with tightened validation"""

    def __init__(self, roi_polygon,
                 # Core parameters
                 yolo_confidence=0.08,
                 roi_entry_threshold=0.1,
                 tracking_zone_radius=150,
                 min_track_length=3,
                 max_distance_for_matching=80,

                 # Recovery system
                 track_recovery_enabled=True,
                 max_frames_to_recover=5,
                 recovery_distance_threshold=120,
                 recovery_confidence_threshold=0.6,

                 # Post-entry validation system
                 validation_delay_frames=8,
                 validation_enabled=True,

                 # Tightened validation parameters
                 frames_to_disappear_threshold=4,        # Must disappear within 2 frames (was 5)
                 movement_away_threshold=20,             # Moved away by 10+ pixels
                 far_from_chimney_threshold=100,          # Now far from chimney (was 120)
                 horizontal_movement_ratio=2.5,          # Horizontal > vertical * 1.5 (was 2.0)
                 significant_horizontal_movement=25,     # Significant horizontal movement threshold
                 close_to_roi_threshold=80,              # Very close to chimney threshold
                 min_close_positions=2,                  # At least 2 out of 3 positions close
                 reasonable_track_min_length=3,          # Minimum reasonable track length
                 reasonable_track_max_length=80,         # Maximum reasonable track length
                 avg_distance_threshold=150,              # Average distance to ROI threshold

                 # Debug
                 debug_mode=True):

        # Core setup
        self.roi_polygon = roi_polygon
        self.bird_tracks = {}
        self.next_track_id = 1
        self.birds_entered_roi = set()
        self.frame_count = 0

        # Post-entry validation system
        self.pending_validation_birds = {}  # Birds waiting for behavior validation
        self.validation_delay_frames = validation_delay_frames
        self.validation_enabled = validation_enabled
        self.validation_stats = {'validated_real': 0, 'rejected_false': 0}

        # Core parameters
        self.yolo_confidence = yolo_confidence
        self.roi_entry_threshold = roi_entry_threshold
        self.tracking_zone_radius = tracking_zone_radius
        self.min_track_length = min_track_length
        self.max_distance_for_matching = max_distance_for_matching

        # Recovery system parameters
        self.track_recovery_enabled = track_recovery_enabled
        self.max_frames_to_recover = max_frames_to_recover
        self.recovery_distance_threshold = recovery_distance_threshold
        self.recovery_confidence_threshold = recovery_confidence_threshold

        # Tightened validation parameters
        self.frames_to_disappear_threshold = frames_to_disappear_threshold
        self.movement_away_threshold = movement_away_threshold
        self.far_from_chimney_threshold = far_from_chimney_threshold
        self.horizontal_movement_ratio = horizontal_movement_ratio
        self.significant_horizontal_movement = significant_horizontal_movement
        self.close_to_roi_threshold = close_to_roi_threshold
        self.min_close_positions = min_close_positions
        self.reasonable_track_min_length = reasonable_track_min_length
        self.reasonable_track_max_length = reasonable_track_max_length
        self.avg_distance_threshold = avg_distance_threshold

        self.debug_mode = debug_mode

        # ROI bounds calculation
        self.roi_min_x, self.roi_min_y = np.min(self.roi_polygon, axis=0)
        self.roi_max_x, self.roi_max_y = np.max(self.roi_polygon, axis=0)
        self.roi_center = ((self.roi_min_x + self.roi_max_x) // 2,
                          (self.roi_min_y + self.roi_max_y) // 2)

        # Recovery system
        self.recently_lost_tracks = {}
        self.recovery_stats = {'recoveries_attempted': 0, 'recoveries_successful': 0}

        print(f"🎯 Bird Tracker initialized for multiple bird detection:")
        print(f"   YOLO confidence: {self.yolo_confidence}")
        print(f"   ROI rectangle: ({self.roi_min_x},{self.roi_min_y}) to ({self.roi_max_x},{self.roi_max_y})")
        print(f"   Tracking zone: {self.tracking_zone_radius}px radius")
        print(f"   Track Recovery: {'ENABLED' if self.track_recovery_enabled else 'DISABLED'}")
        print(f"   Post-Entry Validation: {'ENABLED' if self.validation_enabled else 'DISABLED'}")
        if self.validation_enabled:
            print(f"     Validation delay: {self.validation_delay_frames} frames")
            print(f"     Disappear threshold: {self.frames_to_disappear_threshold} frames")
            print(f"     Movement away threshold: {self.movement_away_threshold}px")
            print(f"     Horizontal movement ratio: {self.horizontal_movement_ratio}")
            print(f"   🔍 TIGHTENED validation parameters loaded")

    def get_mask_centroid(self, mask):
        """Calculate centroid of segmentation mask"""
        y_coords, x_coords = np.where(mask > 0)
        if len(x_coords) == 0:
            return None
        return (int(np.mean(x_coords)), int(np.mean(y_coords)))

    def calculate_distance(self, point1, point2):
        """Calculate Euclidean distance between two points"""
        return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

    def is_point_in_roi_rectangle(self, point):
        """Check if point is inside ROI rectangle"""
        x, y = point
        return (self.roi_min_x <= x <= self.roi_max_x and
                self.roi_min_y <= y <= self.roi_max_y)

    def is_in_tracking_zone(self, centroid):
        """Check if bird is within tracking zone"""
        distance_to_center = self.calculate_distance(centroid, self.roi_center)
        return distance_to_center <= self.tracking_zone_radius

    def simple_roi_detection(self, bird_mask):
        """Simplified ROI detection for multiple birds"""
        try:
            y_coords, x_coords = np.where(bird_mask > 0)
            if len(x_coords) == 0:
                return False, 0.0

            # Check if any pixels overlap with ROI rectangle
            roi_pixels = 0
            for i in range(0, len(x_coords), 2):  # Sample every 2nd pixel for speed
                x, y = x_coords[i], y_coords[i]
                if self.is_point_in_roi_rectangle((x, y)):
                    roi_pixels += 1

            if roi_pixels > 0:
                return True, 1.0

            # Check centroid with small buffer
            centroid_x = int(np.mean(x_coords))
            centroid_y = int(np.mean(y_coords))

            buffer = 5
            for dx in range(-buffer, buffer + 1):
                for dy in range(-buffer, buffer + 1):
                    test_point = (centroid_x + dx, centroid_y + dy)
                    if self.is_point_in_roi_rectangle(test_point):
                        return True, 0.8

            return False, 0.0

        except Exception as e:
            if self.debug_mode:
                print(f"ROI detection error: {e}")
            return False, 0.0

    def get_sam2_mask_id(self, mask, mask_index=None):
        """Generate stable mask ID"""
        if mask_index is not None:
            return f"sam2_mask_{mask_index}"
        return None

    def predict_next_position(self, track_data):
        """Simple position prediction for recovery"""
        centroid_history = track_data.get('centroid_history', [])

        if len(centroid_history) < 2:
            return centroid_history[-1] if centroid_history else None

        # Linear prediction from last 2 positions
        last_pos = centroid_history[-1]
        prev_pos = centroid_history[-2]

        vx = last_pos[0] - prev_pos[0]
        vy = last_pos[1] - prev_pos[1]

        predicted_pos = (int(last_pos[0] + vx), int(last_pos[1] + vy))
        return predicted_pos

    # ============================================================================
    # TIGHTENED POST-ENTRY VALIDATION SYSTEM
    # ============================================================================

    def bird_track_ended_soon_after(self, bird_id, entry_frame):
        """Check if bird track ended very quickly after entry (TIGHTENED)"""
        if bird_id not in self.bird_tracks and bird_id not in self.recently_lost_tracks:
            # Bird completely disappeared - likely real entry
            return True

        # Check if bird is in recently lost tracks
        if bird_id in self.recently_lost_tracks:
            lost_info = self.recently_lost_tracks[bird_id]
            last_seen = lost_info['lost_frame'] - 1  # lost_frame is frame after last seen
            frames_since_entry = last_seen - entry_frame
            # TIGHTENED: Must disappear within configured threshold (default: 2 frames)
            return frames_since_entry <= self.frames_to_disappear_threshold

        return False

    def bird_continued_moving_away(self, bird_id, entry_frame):
        """Check if bird moved away from chimney after 'entry' (ENHANCED)"""
        # Get track data
        track_data = None
        if bird_id in self.bird_tracks:
            track_data = self.bird_tracks[bird_id]
        elif bird_id in self.recently_lost_tracks:
            track_data = self.recently_lost_tracks[bird_id]['track_data']

        if not track_data:
            return False

        centroid_history = track_data.get('centroid_history', [])

        # Find positions after entry frame (relaxed requirement)
        positions_after_entry = []
        entry_position = None

        # Get the entry position and subsequent positions
        if len(centroid_history) >= 2:
            # Assume the last few positions include the entry and after
            for i in range(max(0, len(centroid_history) - 6), len(centroid_history)):
                pos = centroid_history[i]
                if i == len(centroid_history) - 6 + 1:  # Approximate entry position
                    entry_position = pos
                elif entry_position:
                    positions_after_entry.append(pos)

        # TIGHTENED: Need only 1+ positions after entry (was 3+)
        if len(positions_after_entry) >= 1 and entry_position:
            end_pos = positions_after_entry[-1]

            # Check if bird is moving away from chimney center
            entry_distance = self.calculate_distance(entry_position, self.roi_center)
            end_distance = self.calculate_distance(end_pos, self.roi_center)

            # TIGHTENED: If bird moved away from chimney AND is now far
            if (end_distance > entry_distance + self.movement_away_threshold and  # Moving away
                end_distance > self.far_from_chimney_threshold):  # Now far from chimney
                return True  # Moving away behavior detected

            # ADDITIONAL: Check if bird moved mostly horizontally
            horizontal_movement = abs(end_pos[0] - entry_position[0])
            vertical_movement = abs(end_pos[1] - entry_position[1])

            # TIGHTENED: More sensitive horizontal movement detection
            if (horizontal_movement > vertical_movement * self.horizontal_movement_ratio and
                horizontal_movement > self.significant_horizontal_movement):
                return True  # Flying past behavior detected

        return False

    def bird_stayed_near_chimney(self, bird_id, entry_frame):
        """Check if bird genuinely stayed near chimney (NEW CHECK)"""
        # Get track data
        track_data = None
        if bird_id in self.bird_tracks:
            track_data = self.bird_tracks[bird_id]
        elif bird_id in self.recently_lost_tracks:
            track_data = self.recently_lost_tracks[bird_id]['track_data']

        if not track_data:
            return False

        centroid_history = track_data.get('centroid_history', [])

        # Check if bird stayed close to ROI center for multiple frames
        if len(centroid_history) >= 3:
            recent_positions = centroid_history[-3:]  # Last 3 positions
            close_to_roi_count = 0

            for pos in recent_positions:
                distance_to_roi = self.calculate_distance(pos, self.roi_center)
                if distance_to_roi <= self.close_to_roi_threshold:  # Very close to chimney
                    close_to_roi_count += 1

            # If bird stayed close to chimney for most recent positions
            if close_to_roi_count >= self.min_close_positions:  # At least 2 out of 3 positions
                return True

        return False

    def validate_entry_by_behavior(self, bird_id, entry_frame):
        """TIGHTENED validation - more skeptical approach"""

        # PRIORITY 1: Check if bird disappeared very quickly (STRONG evidence of real entry)
        if self.bird_track_ended_soon_after(bird_id, entry_frame):
            if self.debug_mode:
                print(f"✅ Bird #{bird_id} disappeared quickly after entry - REAL ENTRY")
            return True

        # PRIORITY 2: Check if bird genuinely stayed near chimney (GOOD evidence)
        if self.bird_stayed_near_chimney(bird_id, entry_frame):
            if self.debug_mode:
                print(f"✅ Bird #{bird_id} stayed near chimney - REAL ENTRY")
            return True

        # PRIORITY 3: Check if bird moved away/horizontally (STRONG evidence of false positive)
        if self.bird_continued_moving_away(bird_id, entry_frame):
            if self.debug_mode:
                print(f"❌ Bird #{bird_id} moved away from chimney - FALSE POSITIVE")
            return False

        # TIGHTENED DEFAULT: If uncertain, be more skeptical
        # Only keep birds that have some positive evidence
        track_data = None
        if bird_id in self.bird_tracks:
            track_data = self.bird_tracks[bird_id]
        elif bird_id in self.recently_lost_tracks:
            track_data = self.recently_lost_tracks[bird_id]['track_data']

        if track_data:
            centroid_history = track_data.get('centroid_history', [])

            # FINAL CHECK: Bird must have reasonable track length and proximity
            if (len(centroid_history) >= self.reasonable_track_min_length and  # Decent track length
                len(centroid_history) <= self.reasonable_track_max_length):    # Not too long (suggests flying past)

                # Check average distance to ROI
                total_distance = 0
                positions_to_check = centroid_history[-5:]  # Last 5 positions
                for pos in positions_to_check:
                    total_distance += self.calculate_distance(pos, self.roi_center)
                avg_distance = total_distance / len(positions_to_check)

                if avg_distance <= self.avg_distance_threshold:  # Average close to chimney
                    if self.debug_mode:
                        print(f"⚪ Bird #{bird_id} has reasonable track (len={len(centroid_history)}, avg_dist={avg_distance:.1f}) - keeping count")
                    return True

        # DEFAULT: Reject if no positive evidence
        if self.debug_mode:
            track_info = f"len={len(track_data.get('centroid_history', []))}" if track_data else "no_data"
            print(f"❌ Bird #{bird_id} insufficient evidence for real entry ({track_info}) - REJECTED")
        return False

    def process_pending_validations(self):
        """Process any birds ready for validation"""
        birds_to_validate = []

        # Find birds ready for validation
        for bird_id, validation_info in self.pending_validation_birds.items():
            if self.frame_count >= validation_info['validation_frame']:
                birds_to_validate.append(bird_id)

        # Validate each ready bird
        for bird_id in birds_to_validate:
            validation_info = self.pending_validation_birds[bird_id]
            entry_frame = validation_info['entry_frame']

            if self.validate_entry_by_behavior(bird_id, entry_frame):
                # Real entry - add to confirmed count
                self.birds_entered_roi.add(bird_id)
                self.validation_stats['validated_real'] += 1
                if self.debug_mode:
                    print(f"🐦 Bird #{bird_id} VALIDATED - added to final count")
            else:
                # False positive - don't add to count
                self.validation_stats['rejected_false'] += 1
                if self.debug_mode:
                    print(f"❌ Bird #{bird_id} REJECTED - removed from count")

            # Remove from pending
            del self.pending_validation_birds[bird_id]

    def attempt_track_recovery(self, unmatched_detections):
        """Track recovery system for handling multiple birds"""
        if not self.track_recovery_enabled or not self.recently_lost_tracks:
            return unmatched_detections, []

        recovered_tracks = []
        still_unmatched = []

        for detection in unmatched_detections:
            detection_centroid = detection['centroid']
            best_recovery_score = 0.0
            best_recovery_track_id = None

            for lost_track_id, lost_info in self.recently_lost_tracks.items():
                frames_since_lost = self.frame_count - lost_info['lost_frame']

                if frames_since_lost > self.max_frames_to_recover:
                    continue

                predicted_pos = self.predict_next_position(lost_info['track_data'])
                if predicted_pos:
                    distance = self.calculate_distance(detection_centroid, predicted_pos)

                    if distance <= self.recovery_distance_threshold:
                        score = 1.0 - (distance / self.recovery_distance_threshold)
                        score *= (1.0 - (frames_since_lost / self.max_frames_to_recover))

                        if score > best_recovery_score and score >= self.recovery_confidence_threshold:
                            best_recovery_score = score
                            best_recovery_track_id = lost_track_id

            if best_recovery_track_id is not None:
                # Recover the track
                lost_info = self.recently_lost_tracks[best_recovery_track_id]
                recovered_track_data = lost_info['track_data'].copy()

                recovered_track_data['centroid_history'].append(detection_centroid)
                recovered_track_data['last_seen'] = self.frame_count
                recovered_track_data['sam2_mask_id'] = detection.get('sam2_mask_id')

                self.bird_tracks[best_recovery_track_id] = recovered_track_data
                del self.recently_lost_tracks[best_recovery_track_id]

                self.recovery_stats['recoveries_attempted'] += 1
                self.recovery_stats['recoveries_successful'] += 1

                recovered_tracks.append({
                    'track_id': best_recovery_track_id,
                    'detection': detection
                })

                if self.debug_mode:
                    gap_frames = self.frame_count - lost_info['lost_frame']
                    print(f"🔄 RECOVERED Track #{best_recovery_track_id} after {gap_frames} frames")
            else:
                still_unmatched.append(detection)

        return still_unmatched, recovered_tracks

    def update_recently_lost_tracks(self):
        """Update recently lost tracks buffer"""
        tracks_to_move = []
        for track_id, track_data in self.bird_tracks.items():
            frames_since_seen = self.frame_count - track_data['last_seen']
            if frames_since_seen >= 1:
                tracks_to_move.append(track_id)

        for track_id in tracks_to_move:
            if track_id not in self.recently_lost_tracks:
                self.recently_lost_tracks[track_id] = {
                    'track_data': self.bird_tracks[track_id].copy(),
                    'lost_frame': self.bird_tracks[track_id]['last_seen'] + 1
                }
            del self.bird_tracks[track_id]

        # Clean up old tracks
        old_tracks = []
        for track_id, lost_info in self.recently_lost_tracks.items():
            frames_since_lost = self.frame_count - lost_info['lost_frame']
            if frames_since_lost > self.max_frames_to_recover:
                old_tracks.append(track_id)

        for track_id in old_tracks:
            del self.recently_lost_tracks[track_id]

    def update_tracks(self, bird_detections):
        """Main tracking update for multiple birds with post-entry validation"""
        self.frame_count += 1
        self.update_recently_lost_tracks()

        # Process any pending validations first
        if self.validation_enabled:
            self.process_pending_validations()

        # Filter detections to tracking zone
        zone_detections = []
        birds_in_roi_this_frame = 0

        for i, detection in enumerate(bird_detections):
            mask = detection['mask']
            confidence = float(detection['confidence'])
            sam2_mask_id = detection.get('sam2_mask_id', self.get_sam2_mask_id(mask, mask_index=i))
            centroid = self.get_mask_centroid(mask)

            if centroid and self.is_in_tracking_zone(centroid):
                in_roi, roi_confidence = self.simple_roi_detection(mask)
                if in_roi:
                    birds_in_roi_this_frame += 1

                zone_detections.append({
                    'centroid': centroid,
                    'mask': mask,
                    'confidence': confidence,
                    'roi_confidence': roi_confidence,
                    'in_roi': in_roi,
                    'sam2_mask_id': sam2_mask_id
                })

        # Track matching
        matched_tracks = set()
        unmatched_detections = []

        for curr_data in zone_detections:
            curr_centroid = curr_data['centroid']
            sam2_mask_id = curr_data['sam2_mask_id']
            best_match_id = None
            best_score = 0.0

            # Try SAM2 ID matching first
            for track_id, track_data in self.bird_tracks.items():
                if track_id in matched_tracks:
                    continue

                if (track_data.get('sam2_mask_id') == sam2_mask_id and
                    sam2_mask_id is not None):
                    best_match_id = track_id
                    best_score = 1.0
                    break

            # Distance matching as fallback
            if best_match_id is None:
                for track_id, track_data in self.bird_tracks.items():
                    if track_id in matched_tracks:
                        continue

                    last_centroid = track_data['centroid_history'][-1]
                    distance = self.calculate_distance(curr_centroid, last_centroid)

                    if distance <= self.max_distance_for_matching:
                        score = 1.0 - (distance / self.max_distance_for_matching)
                        if score > best_score:
                            best_score = score
                            best_match_id = track_id

            if best_match_id is not None and best_score > 0.3:
                # Update existing track
                matched_tracks.add(best_match_id)
                track_data = self.bird_tracks[best_match_id]
                track_data['centroid_history'].append(curr_centroid)
                track_data['last_seen'] = self.frame_count
                track_data['sam2_mask_id'] = sam2_mask_id

                # Check for ROI entry - MODIFIED FOR VALIDATION SYSTEM
                if not track_data.get('entered_roi', False) and best_match_id not in self.pending_validation_birds:
                    roi_entered = (curr_data['in_roi'] and
                                  curr_data['roi_confidence'] >= self.roi_entry_threshold and
                                  len(track_data['centroid_history']) >= self.min_track_length)

                    if roi_entered:
                        track_data['entered_roi'] = True
                        track_data['entry_location'] = curr_centroid
                        track_data['entry_frame'] = self.frame_count

                        if self.validation_enabled:
                            # Add to pending validation instead of immediate count
                            self.pending_validation_birds[best_match_id] = {
                                'entry_frame': self.frame_count,
                                'validation_frame': self.frame_count + self.validation_delay_frames
                            }
                            print(f"🔄 Bird #{best_match_id} entered ROI at frame {self.frame_count} - PENDING VALIDATION")
                            print(f"   Entry location: {curr_centroid}")
                            print(f"   Track length: {len(track_data['centroid_history'])} frames")
                            print(f"   Will validate at frame: {self.frame_count + self.validation_delay_frames}")
                        else:
                            # Original immediate counting (if validation disabled)
                            self.birds_entered_roi.add(best_match_id)
                            print(f"🐦 Bird #{best_match_id} entered ROI at frame {self.frame_count}")
                            print(f"   Entry location: {curr_centroid}")
                            print(f"   Track length: {len(track_data['centroid_history'])} frames")
            else:
                unmatched_detections.append(curr_data)

        # Track recovery
        still_unmatched, recovered_tracks = self.attempt_track_recovery(unmatched_detections)

        # Process recovered tracks for ROI entry - MODIFIED FOR VALIDATION SYSTEM
        for recovery_info in recovered_tracks:
            track_id = recovery_info['track_id']
            detection = recovery_info['detection']
            track_data = self.bird_tracks[track_id]

            if not track_data.get('entered_roi', False) and track_id not in self.pending_validation_birds:
                roi_entered = (detection['in_roi'] and
                              detection['roi_confidence'] >= self.roi_entry_threshold and
                              len(track_data['centroid_history']) >= self.min_track_length)

                if roi_entered:
                    track_data['entered_roi'] = True
                    track_data['entry_location'] = detection['centroid']
                    track_data['entry_frame'] = self.frame_count

                    if self.validation_enabled:
                        # Add to pending validation
                        self.pending_validation_birds[track_id] = {
                            'entry_frame': self.frame_count,
                            'validation_frame': self.frame_count + self.validation_delay_frames
                        }
                        print(f"🔄 RECOVERED Bird #{track_id} entered ROI at frame {self.frame_count} - PENDING VALIDATION")
                    else:
                        # Original immediate counting
                        self.birds_entered_roi.add(track_id)
                        print(f"🐦 RECOVERED Bird #{track_id} entered ROI at frame {self.frame_count}")

        # Create new tracks
        for new_data in still_unmatched:
            track_id = self.next_track_id

            self.bird_tracks[track_id] = {
                'centroid_history': [new_data['centroid']],
                'entered_roi': False,
                'last_seen': self.frame_count,
                'sam2_mask_id': new_data['sam2_mask_id']
            }

            # Immediate ROI entry for new tracks (if min_track_length is 1) - MODIFIED FOR VALIDATION
            if (new_data['in_roi'] and
                new_data['roi_confidence'] >= self.roi_entry_threshold and
                self.min_track_length <= 1):

                self.bird_tracks[track_id]['entered_roi'] = True
                self.bird_tracks[track_id]['entry_location'] = new_data['centroid']
                self.bird_tracks[track_id]['entry_frame'] = self.frame_count

                if self.validation_enabled:
                    # Add to pending validation
                    self.pending_validation_birds[track_id] = {
                        'entry_frame': self.frame_count,
                        'validation_frame': self.frame_count + self.validation_delay_frames
                    }
                    print(f"🔄 NEW Bird #{track_id} entered ROI at frame {self.frame_count} - PENDING VALIDATION")
                else:
                    # Original immediate counting
                    self.birds_entered_roi.add(track_id)
                    print(f"🐦 NEW Bird #{track_id} entered ROI at frame {self.frame_count}")

            self.next_track_id += 1

        # Debug output for multiple birds - UPDATED FOR VALIDATION SYSTEM
        if birds_in_roi_this_frame > 0 or len(zone_detections) > 0:
            active_tracks = len(self.bird_tracks)
            lost_tracks = len(self.recently_lost_tracks)
            confirmed_birds = len(self.birds_entered_roi)
            pending_birds = len(self.pending_validation_birds)

            print(f"🔍 Frame {self.frame_count}: {len(zone_detections)} detections, "
                  f"{birds_in_roi_this_frame} in ROI, {active_tracks} active, "
                  f"{lost_tracks} lost, {confirmed_birds} confirmed, {pending_birds} pending validation")

    def process_end_of_video(self):
        """End-of-video processing with validation system"""
        print(f"\n🏁 End-of-video processing - checking {len(self.bird_tracks)} tracks...")

        # First, process any remaining pending validations
        if self.validation_enabled and self.pending_validation_birds:
            print(f"🔄 Processing {len(self.pending_validation_birds)} pending validations...")

            # Force validation of all pending birds
            for bird_id, validation_info in list(self.pending_validation_birds.items()):
                entry_frame = validation_info['entry_frame']

                if self.validate_entry_by_behavior(bird_id, entry_frame):
                    self.birds_entered_roi.add(bird_id)
                    self.validation_stats['validated_real'] += 1
                    print(f"🐦 FINAL: Bird #{bird_id} validated and counted")
                else:
                    self.validation_stats['rejected_false'] += 1
                    print(f"❌ FINAL: Bird #{bird_id} rejected as false positive")

                # Remove from pending
                del self.pending_validation_birds[bird_id]

        # Then, check remaining unvalidated tracks (original end-of-video logic)
        final_counts_added = 0
        for track_id, track_data in self.bird_tracks.items():
            if not track_data.get('entered_roi', False) and track_id not in self.birds_entered_roi:
                # Check recent frames for ROI presence
                recent_frames = min(3, len(track_data['centroid_history']))
                roi_hits = 0

                for i in range(-recent_frames, 0):
                    if abs(i) <= len(track_data['centroid_history']):
                        centroid_idx = len(track_data['centroid_history']) + i
                        if centroid_idx >= 0:
                            centroid = track_data['centroid_history'][centroid_idx]
                            if self.is_point_in_roi_rectangle(centroid):
                                roi_hits += 1

                if roi_hits >= 1:  # Any ROI hit in recent frames
                    if self.validation_enabled:
                        # Apply validation to these end-of-video birds too
                        track_data['entered_roi'] = True
                        track_data['entry_location'] = track_data['centroid_history'][-1]
                        track_data['entry_frame'] = self.frame_count

                        # Validate immediately since we're at end of video
                        if self.validate_entry_by_behavior(track_id, self.frame_count):
                            self.birds_entered_roi.add(track_id)
                            final_counts_added += 1
                            print(f"🐦 FINAL: Bird #{track_id} counted (was in ROI {roi_hits}/{recent_frames} recent frames) - VALIDATED")
                        else:
                            print(f"❌ FINAL: Bird #{track_id} rejected (was in ROI {roi_hits}/{recent_frames} recent frames) - FAILED VALIDATION")
                    else:
                        # Original logic without validation
                        track_data['entered_roi'] = True
                        track_data['entry_location'] = track_data['centroid_history'][-1]
                        track_data['entry_frame'] = self.frame_count
                        self.birds_entered_roi.add(track_id)
                        final_counts_added += 1
                        print(f"🐦 FINAL: Bird #{track_id} counted (was in ROI {roi_hits}/{recent_frames} recent frames)")

        if final_counts_added > 0:
            print(f"✅ Added {final_counts_added} birds from end-of-video processing")

        # Display validation statistics
        if self.validation_enabled:
            total_validated = self.validation_stats['validated_real']
            total_rejected = self.validation_stats['rejected_false']
            total_processed = total_validated + total_rejected

            print(f"\n📊 TIGHTENED VALIDATION SUMMARY:")
            print(f"   Total birds processed for validation: {total_processed}")
            print(f"   Validated as real entries: {total_validated}")
            print(f"   Rejected as false positives: {total_rejected}")
            if total_processed > 0:
                rejection_rate = (total_rejected / total_processed) * 100
                print(f"   False positive rejection rate: {rejection_rate:.1f}%")
            print(f"   🔍 Validation parameters:")
            print(f"     Disappear threshold: {self.frames_to_disappear_threshold} frames")
            print(f"     Movement away threshold: {self.movement_away_threshold}px")
            print(f"     Horizontal movement ratio: {self.horizontal_movement_ratio}")

        return final_counts_added

    def get_count(self):
        """Get total count of birds that entered ROI"""
        return len(self.birds_entered_roi)

    def get_tracking_info(self):
        """Get tracking information including validation stats"""
        info = {
            'total_birds_entered': len(self.birds_entered_roi),
            'active_tracks': len(self.bird_tracks),
            'recently_lost_tracks': len(self.recently_lost_tracks),
            'confirmed_entries': list(self.birds_entered_roi),
            'recovery_stats': self.recovery_stats.copy()
        }

        if self.validation_enabled:
            info['pending_validation'] = len(self.pending_validation_birds)
            info['validation_stats'] = self.validation_stats.copy()

        return info

Detection and processing

In [14]:
# ============================================================================
# DETECTION AND PROCESSING FUNCTIONS
# ============================================================================

def conservative_yolo_detection(model, frame, device, confidence=0.08, iou=0.4, max_det=100):
    """YOLO detection with optimized parameters"""
    results = model.predict(
        source=frame,
        device=device,
        conf=confidence,
        iou=iou,
        max_det=max_det,
        verbose=False
    )[0]
    return results

def aggressive_bird_counting_pipeline(video_path, output_suffix=""):
    """Main processing pipeline optimized for multiple birds with validation"""

    # Generate output names
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    output_video_path = f"{PATH}/data/sam2_output_{base_name}{output_suffix}.mp4"
    output_json_path = f"{PATH}/data/sam2_results_{base_name}{output_suffix}.json"

    # Load ROI from saved file
    roi_path = f"{PATH}/data/roi_polygon.npy"
    roi_polygon = np.load(roi_path)

    # Initialize tracker with validation DISABLED
    bird_tracker = AggressiveBirdTracker(roi_polygon, validation_enabled=True)

    # Load models
    if not os.path.exists("yolo11x.pt"):
        print("📥 Downloading YOLO model...")
        import subprocess
        subprocess.run(["wget", "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11x.pt"],
                      check=True, capture_output=True)

    yolo = YOLO("yolo11x.pt")
    sam = SAM("sam2_b.pt")
    device = 0 if torch.cuda.is_available() else "cpu"

    # Video setup
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print(f"🎯 PROCESSING FOR MULTIPLE BIRDS (VALIDATION DISABLED)")
    print(f"🎯 Video info: {frame_count} total frames at {fps:.1f} fps")

    # Main processing loop
    all_results = []
    frame_idx = 0

    with tqdm(total=frame_count, desc="Processing multiple birds") as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            try:
                vis_frame = frame.copy()

                # Draw ROI
                cv2.polylines(vis_frame, [roi_polygon], isClosed=True, color=(0, 255, 255), thickness=2)
                cv2.putText(vis_frame, "CHIMNEY ROI", (roi_polygon[0][0], roi_polygon[0][1] - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                # YOLO detection
                yolo_result = conservative_yolo_detection(yolo, frame, device)

                # Filter for birds
                bird_boxes = []
                bird_confidences = []

                if yolo_result and yolo_result.boxes is not None:
                    for box in yolo_result.boxes:
                        try:
                            cls_id = int(box.cls)
                            confidence = float(box.conf.item()) if hasattr(box.conf, 'item') else float(box.conf)
                            class_name = yolo.names[cls_id]

                            if class_name == 'bird' and confidence >= 0.05:
                                bird_boxes.append(box.xyxy[0].cpu().numpy().tolist())
                                bird_confidences.append(confidence)
                        except:
                            continue

                bird_detections = []

                if bird_boxes:
                    try:
                        # SAM segmentation
                        sam_result = sam.predict(source=frame, bboxes=bird_boxes, device=device, verbose=False)[0]

                        if sam_result.masks is not None:
                            masks = sam_result.masks.data.cpu().numpy()

                            for i, mask in enumerate(masks):
                                try:
                                    confidence = bird_confidences[i]
                                    sam2_mask_id = bird_tracker.get_sam2_mask_id(mask, mask_index=i)

                                    bird_detections.append({
                                        'mask': mask,
                                        'confidence': confidence,
                                        'bbox': bird_boxes[i],
                                        'frame_idx': frame_idx,
                                        'sam2_mask_id': sam2_mask_id
                                    })

                                    # Visualization
                                    color = [int(c) for c in np.random.randint(80, 220, 3)]
                                    mask_img = (mask * 255).astype(np.uint8)
                                    contours, _ = cv2.findContours(mask_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                                    if contours:
                                        cv2.drawContours(vis_frame, contours, -1, color, 2)
                                        centroid = bird_tracker.get_mask_centroid(mask)
                                        if centroid and 0 <= centroid[0] < width and 0 <= centroid[1] < height:
                                            cv2.putText(vis_frame, f"{confidence:.2f}",
                                                       (centroid[0] + 5, centroid[1] - 5),
                                                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
                                except:
                                    continue
                    except Exception as e:
                        print(f"SAM error at frame {frame_idx}: {e}")

                # Update tracking
                bird_tracker.update_tracks(bird_detections)

                # Get current count
                tracking_info = bird_tracker.get_tracking_info()
                current_count = tracking_info['total_birds_entered']

                # Text overlay - UPDATED FOR VALIDATION
                overlay = vis_frame.copy()
                cv2.rectangle(overlay, (5, 5), (450, 150), (0, 0, 0), -1)
                vis_frame = cv2.addWeighted(vis_frame, 0.8, overlay, 0.2, 0)

                cv2.putText(vis_frame, f"BIRDS COUNTED: {current_count}",
                           (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

                cv2.putText(vis_frame, f"Frame: {frame_idx:03d}/{frame_count} | Detected: {len(bird_detections):02d}",
                           (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

                cv2.putText(vis_frame, f"Active: {tracking_info['active_tracks']:02d} | Lost: {tracking_info['recently_lost_tracks']:02d}",
                           (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

                # Show validation info if enabled
                if bird_tracker.validation_enabled:
                    pending_count = tracking_info.get('pending_validation', 0)
                    cv2.putText(vis_frame, f"Pending validation: {pending_count:02d}",
                               (10, 115), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
                else:
                    cv2.putText(vis_frame, f"Validation: DISABLED",
                               (10, 115), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (128, 128, 128), 1)

                # Show counted birds as green dots (and pending birds as yellow dots if validation enabled)
                for track_id, track_data in bird_tracker.bird_tracks.items():
                    if track_data.get('entered_roi', False) and 'entry_location' in track_data:
                        entry_point = track_data['entry_location']
                        if (entry_point and len(entry_point) == 2 and
                            0 <= entry_point[0] < width and 0 <= entry_point[1] < height):

                            if bird_tracker.validation_enabled:
                                # Different colors for confirmed vs pending (validation enabled)
                                if track_id in bird_tracker.birds_entered_roi:
                                    color = (0, 255, 0)  # Green for confirmed
                                    label = f"#{track_id}"
                                elif track_id in bird_tracker.pending_validation_birds:
                                    color = (0, 255, 255)  # Yellow for pending
                                    label = f"#{track_id}?"
                                else:
                                    continue  # Skip if neither confirmed nor pending
                            else:
                                # All birds are immediately confirmed (validation disabled)
                                if track_id in bird_tracker.birds_entered_roi:
                                    color = (0, 255, 0)  # Green for all counted birds
                                    label = f"#{track_id}"
                                else:
                                    continue  # Skip if not counted

                            cv2.circle(vis_frame, tuple(entry_point), 3, color, -1)
                            cv2.putText(vis_frame, label,
                                       (entry_point[0] + 6, entry_point[1] - 3),
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1)

                # Save frame data
                frame_results = {
                    'frame_idx': frame_idx,
                    'total_count': current_count,
                    'birds_detected': len(bird_detections),
                    'active_tracks': tracking_info['active_tracks']
                }
                all_results.append(frame_results)

                out.write(vis_frame)
                frame_idx += 1
                pbar.update(1)

            except Exception as e:
                print(f"Error at frame {frame_idx}: {e}")
                frame_idx += 1
                pbar.update(1)
                continue

    cap.release()
    out.release()

    # End processing
    final_birds_added = bird_tracker.process_end_of_video()
    final_count = bird_tracker.get_count()
    final_tracking_info = bird_tracker.get_tracking_info()

    # Results - UPDATED FOR VALIDATION
    final_results = {
        'summary': {
            'video_file': os.path.basename(video_path),
            'total_frames_processed': frame_idx,
            'total_birds_counted': final_count,
            'birds_added_at_end': final_birds_added,
            'method': 'simplified_multiple_bird_tracking_no_validation',
            'validation_enabled': bird_tracker.validation_enabled
        },
        'frame_by_frame': all_results,
        'roi_polygon': roi_polygon.tolist()
    }

    # Add validation stats if enabled
    if bird_tracker.validation_enabled:
        final_results['summary']['validation_stats'] = final_tracking_info.get('validation_stats', {})

    # Save results JSON file (FIXED - suppress printing)
    try:
        with open(output_json_path, "w") as f:
            json.dump(final_results, f, indent=4)
        print(f"📊 Results JSON saved: {output_json_path}")
    except Exception as e:
        print(f"⚠️ Warning: Could not save JSON results: {e}")

    print(f"\n✅ PROCESSING COMPLETE!")
    print(f"🐦 FINAL COUNT: {final_count} birds")
    print(f"📊 End-of-video adds: +{final_birds_added} birds")

    # Show validation summary
    if bird_tracker.validation_enabled:
        validation_stats = final_tracking_info.get('validation_stats', {})
        total_validated = validation_stats.get('validated_real', 0)
        total_rejected = validation_stats.get('rejected_false', 0)
        print(f"✅ Validated as real: {total_validated} birds")
        print(f"❌ Rejected as false: {total_rejected} birds")
        print(f"🔍 Post-entry validation: ENABLED")
    else:
        print(f"🔍 Post-entry validation: DISABLED - All birds counted immediately")
        print(f"⚡ Faster processing: No validation delays")

    print(f"📹 Output video: {output_video_path}")

    return final_results

In [15]:
# ============================================================================
# MAIN TESTING FUNCTIONS
# ============================================================================

def run_segment_test(segment_info, video_url):
    """Run tracker on a single segment"""
    segment_name = segment_info['name']
    start_time = segment_info['start']
    end_time = segment_info['end']
    expected_count = segment_info.get('expected_count', None)

    print(f"\n🔄 TESTING SEGMENT: {segment_name}")
    print(f"⏱️ Time range: {start_time} to {end_time}")
    print("-" * 50)

    try:
        # Download video segment (with caching)
        video_path, video_filename = download_video_segment_unique(
            video_url, start_time, end_time, segment_name
        )

        if not os.path.exists(video_path):
            raise Exception(f"Video file not found: {video_path}")

        # Run processing pipeline
        print(f"🤖 Running tracker on: {video_filename}")
        results = aggressive_bird_counting_pipeline(video_path, output_suffix=f"_{segment_name}")

        actual_count = results['summary']['total_birds_counted']

        # Calculate accuracy
        accuracy_info = {}
        if expected_count:
            accuracy_pct = (actual_count / expected_count) * 100
            difference = actual_count - expected_count
            accuracy_info = {
                'expected': expected_count,
                'actual': actual_count,
                'accuracy_percentage': accuracy_pct,
                'difference': difference,
                'status': 'EXCELLENT' if accuracy_pct >= 90 else 'GOOD' if accuracy_pct >= 80 else 'NEEDS_WORK'
            }

        return {
            'segment_name': segment_name,
            'time_range': f"{start_time} to {end_time}",
            'video_filename': video_filename,
            'model_count': actual_count,
            'accuracy_info': accuracy_info,
            'full_results': results,
            'success': True
        }

    except Exception as e:
        print(f"❌ Error in segment {segment_name}: {e}")
        return {
            'segment_name': segment_name,
            'time_range': f"{start_time} to {end_time}",
            'video_filename': 'N/A',
            'model_count': 0,
            'accuracy_info': {'status': 'ERROR'},
            'error': str(e),
            'success': False
        }

def display_comprehensive_summary(all_results):
    """Display detailed summary of all test results with enhanced validation info"""
    print(f"\n" + "="*85)
    print(f"📊 COMPREHENSIVE TEST RESULTS SUMMARY")
    print(f"="*85)

    print(f"{'Segment':<18} {'Time Range':<15} {'Expected':<10} {'Actual':<10} {'Accuracy':<12} {'Status':<15}")
    print(f"{'-'*18} {'-'*15} {'-'*10} {'-'*10} {'-'*12} {'-'*15}")

    total_expected = 0
    total_actual = 0
    successful_tests = 0

    # Track validation statistics across all segments
    total_validated = 0
    total_rejected = 0
    validation_enabled_count = 0
    all_rejected_birds = []

    for result in all_results:
        segment_name = result['segment_name'][:17]
        time_range = result['time_range'][:14]
        model_count = result['model_count']

        if result['success'] and result['accuracy_info'] and 'expected' in result['accuracy_info']:
            expected = result['accuracy_info']['expected']
            accuracy = f"{result['accuracy_info']['accuracy_percentage']:.1f}%"
            status = result['accuracy_info']['status']

            total_expected += expected
            total_actual += model_count
            successful_tests += 1

            print(f"{segment_name:<18} {time_range:<15} {expected:<10} {model_count:<10} {accuracy:<12} {status:<15}")

            # Collect validation stats from this segment
            if result.get('full_results', {}).get('summary', {}).get('validation_enabled', False):
                validation_enabled_count += 1
                validation_stats = result.get('full_results', {}).get('summary', {}).get('validation_stats', {})
                segment_validated = validation_stats.get('validated_real', 0)
                segment_rejected = validation_stats.get('rejected_false', 0)
                total_validated += segment_validated
                total_rejected += segment_rejected

                # Collect rejected birds info
                rejected_birds = validation_stats.get('rejected_birds', [])
                for rejection in rejected_birds:
                    rejection['segment'] = segment_name
                    all_rejected_birds.append(rejection)

        else:
            status = "ERROR" if not result['success'] else "MODEL_ONLY"
            print(f"{segment_name:<18} {time_range:<15} {'N/A':<10} {model_count:<10} {'N/A':<12} {status:<15}")

    # Overall statistics
    if successful_tests > 0:
        overall_accuracy = (total_actual / total_expected) * 100
        print(f"{'-'*18} {'-'*15} {'-'*10} {'-'*10} {'-'*12} {'-'*15}")
        print(f"{'TOTAL':<18} {'ALL':<15} {total_expected:<10} {total_actual:<10} {overall_accuracy:.1f}{'%':<11} {'OVERALL':<15}")

        print(f"\n🎯 FINAL PERFORMANCE ANALYSIS:")
        print(f"   📊 Total birds expected: {total_expected}")
        print(f"   🐦 Total birds detected: {total_actual}")
        print(f"   📈 Overall accuracy: {overall_accuracy:.1f}%")
        print(f"   📉 Total difference: {total_actual - total_expected:+d} birds")

        # Enhanced validation summary
        if validation_enabled_count > 0:
            print(f"\n🔍 VALIDATION SYSTEM SUMMARY:")
            print(f"   Tests with validation enabled: {validation_enabled_count}/{successful_tests}")
            print(f"   Total birds validated as real: {total_validated}")
            print(f"   Total birds rejected as false: {total_rejected}")
            if (total_validated + total_rejected) > 0:
                rejection_rate = (total_rejected / (total_validated + total_rejected)) * 100
                print(f"   False positive rejection rate: {rejection_rate:.1f}%")

            # Show rejected birds summary across all segments
            if all_rejected_birds:
                print(f"\n🚨 REJECTED BIRDS ACROSS ALL SEGMENTS:")
                print("-" * 60)
                for i, rejection in enumerate(all_rejected_birds, 1):
                    bird_id = rejection['bird_id']
                    reason = rejection['reason']
                    segment = rejection.get('segment', 'unknown')
                    frame = rejection.get('frame', 'unknown')

                    print(f"{i}. Bird #{bird_id} in {segment} (Frame {frame})")
                    print(f"   Reason: {reason}")
                    if 'track_length' in rejection:
                        print(f"   Track length: {rejection['track_length']} frames")
                    if 'avg_distance' in rejection:
                        print(f"   Avg distance: {rejection['avg_distance']:.1f}px")
                    print()

        # Performance rating
        if overall_accuracy >= 90:
            print(f"   🏆 Overall Rating: OUTSTANDING! 🥇")
        elif overall_accuracy >= 85:
            print(f"   ⭐ Overall Rating: EXCELLENT! ✨")
        elif overall_accuracy >= 75:
            print(f"   ✅ Overall Rating: GOOD")
        else:
            print(f"   ⚠️ Overall Rating: NEEDS IMPROVEMENT")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_all_tests():
    """Execute complete test suite"""
    print("🚀 STARTING BIRD CLASSIFICATION TESTING")
    print(f"📺 Video Source: {VIDEO_URL}")
    print(f"🧪 Testing {len(segments_to_test)} segments")
    print("=" * 80)

    # Run all tests
    all_test_results = []

    for i, segment in enumerate(segments_to_test, 1):
        print(f"\n🔬 TEST {i}/{len(segments_to_test)}")
        result = run_segment_test(segment, VIDEO_URL)
        all_test_results.append(result)

        # Show immediate results
        if result['success']:
            print(f"✅ {segment['name']}: {result['model_count']} birds detected")
            if result['accuracy_info'] and 'expected' in result['accuracy_info']:
                acc = result['accuracy_info']['accuracy_percentage']
                print(f"   Accuracy: {acc:.1f}% ({result['accuracy_info']['status']})")
        else:
            print(f"❌ {segment['name']}: Test failed")

        print("=" * 50)

    # Display comprehensive results
    display_comprehensive_summary(all_test_results)

    # Save master results file
    master_results = {
        'test_metadata': {
            'test_date': str(datetime.now()),
            'video_url': VIDEO_URL,
            'total_segments': len(segments_to_test),
            'tracker_version': 'enhanced_validation_v1'
        },
        'segment_configuration': segments_to_test,
        'test_results': all_test_results
    }

    master_results_path = f"{PATH}/data/master_test_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

    # Save master results (suppress JSON printing)
    try:
        with open(master_results_path, 'w') as f:
            json.dump(master_results, f, indent=4)
        print(f"\n💾 Master results saved: {master_results_path}")
    except Exception as e:
        print(f"\n⚠️ Warning: Could not save master results: {e}")

    print(f"🎉 COMPLETE TEST SUITE FINISHED!")

    return all_test_results

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

print("✅ Enhanced Bird Classification Pipeline Loaded!")
print("\n🎯 USAGE:")
print("1. run_all_tests() - Execute complete test suite")
print("2. run_segment_test(segments_to_test[0], VIDEO_URL) - Test single segment")
print("\n💡 KEY IMPROVEMENTS:")
print("   • Enhanced rejection tracking with detailed reasons")
print("   • Balanced validation (catches obvious false positives only)")
print("   • No JSON spam in output")
print("   • Comprehensive rejection summary")
print("   • Video caching (no re-downloads)")

# Uncomment to run automatically:
# run_all_tests()

✅ Enhanced Bird Classification Pipeline Loaded!

🎯 USAGE:
1. run_all_tests() - Execute complete test suite
2. run_segment_test(segments_to_test[0], VIDEO_URL) - Test single segment

💡 KEY IMPROVEMENTS:
   • Enhanced rejection tracking with detailed reasons
   • Balanced validation (catches obvious false positives only)
   • No JSON spam in output
   • Comprehensive rejection summary
   • Video caching (no re-downloads)


final

In [16]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_all_tests():
    """Execute complete test suite"""
    print("🚀 STARTING BIRD CLASSIFICATION TESTING")
    print(f"📺 Video Source: {VIDEO_URL}")
    print(f"🧪 Testing {len(segments_to_test)} segments")
    print("=" * 80)

    # Run all tests
    all_test_results = []

    for i, segment in enumerate(segments_to_test, 1):
        print(f"\n🔬 TEST {i}/{len(segments_to_test)}")
        result = run_segment_test(segment, VIDEO_URL)
        all_test_results.append(result)

        # Show immediate results
        if result['success']:
            print(f"✅ {segment['name']}: {result['model_count']} birds detected")
            if result['accuracy_info'] and 'expected' in result['accuracy_info']:
                acc = result['accuracy_info']['accuracy_percentage']
                print(f"   Accuracy: {acc:.1f}% ({result['accuracy_info']['status']})")
        else:
            print(f"❌ {segment['name']}: Test failed")

    # Display comprehensive results
    display_comprehensive_summary(all_test_results)

    # Save master results
    master_results = {
        'test_metadata': {
            'test_date': str(datetime.now()),
            'video_url': VIDEO_URL,
            'total_segments': len(segments_to_test),
            'tracker_version': 'simplified_multiple_bird_v1'
        },
        'segment_configuration': segments_to_test,
        'test_results': all_test_results
    }

    master_results_path = f"{PATH}/data/master_test_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(master_results_path, 'w') as f:
        json.dump(master_results, f, indent=4)

    f.close()

    print(f"\n💾 Master results saved: {master_results_path}")
    print(f"🎉 COMPLETE TEST SUITE FINISHED!")

    return all_test_results

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

print("✅ Cleaned Bird Classification Pipeline Loaded!")
print("\n🎯 USAGE:")
print("1. run_all_tests() - Execute complete test suite")
print("2. run_segment_test(segments_to_test[0], VIDEO_URL) - Test single segment")
print("\n💡 KEY IMPROVEMENTS:")
print("   • Video caching (no re-downloads)")
print("   • Simplified tracking logic")
print("   • Optimized for multiple birds")
print("   • Removed ~60% of unused code")
print("   • Clear function structure")

_ = run_all_tests()

✅ Cleaned Bird Classification Pipeline Loaded!

🎯 USAGE:
1. run_all_tests() - Execute complete test suite
2. run_segment_test(segments_to_test[0], VIDEO_URL) - Test single segment

💡 KEY IMPROVEMENTS:
   • Video caching (no re-downloads)
   • Simplified tracking logic
   • Optimized for multiple birds
   • Removed ~60% of unused code
   • Clear function structure
🚀 STARTING BIRD CLASSIFICATION TESTING
📺 Video Source: https://www.youtube.com/watch?v=mkquDCpPD9c
🧪 Testing 5 segments

🔬 TEST 1/5

🔄 TESTING SEGMENT: segment_1_baseline
⏱️ Time range: 0:05 to 0:16
--------------------------------------------------
✅ Using existing file: downloaded_video_0-05_to_0-16.mp4
🤖 Running tracker on: downloaded_video_0-05_to_0-16.mp4
🎯 Bird Tracker initialized for multiple bird detection:
   YOLO confidence: 0.08
   ROI rectangle: (580,550) to (780,720)
   Tracking zone: 150px radius
   Track Recovery: ENABLED
   Post-Entry Validation: ENABLED
     Validation delay: 8 frames
     Disappear threshold:

Processing multiple birds:   0%|          | 1/264 [00:02<12:19,  2.81s/it]

🔍 Frame 1: 1 detections, 0 in ROI, 1 active, 0 lost, 0 confirmed, 0 pending validation


Processing multiple birds:   1%|          | 2/264 [00:03<06:04,  1.39s/it]

🔄 RECOVERED Track #1 after 0 frames
🔍 Frame 2: 1 detections, 1 in ROI, 1 active, 0 lost, 0 confirmed, 0 pending validation


...

Processing multiple birds: 100%|█████████▉| 431/432 [04:17<00:00,  3.38it/s]

🔄 RECOVERED Track #259 after 0 frames
🔍 Frame 431: 1 detections, 0 in ROI, 1 active, 2 lost, 110 confirmed, 2 pending validation


Processing multiple birds: 100%|██████████| 432/432 [04:17<00:00,  1.68it/s]

🔄 RECOVERED Track #259 after 0 frames
🔄 RECOVERED Bird #259 entered ROI at frame 432 - PENDING VALIDATION
🔍 Frame 432: 1 detections, 1 in ROI, 1 active, 2 lost, 110 confirmed, 3 pending validation

🏁 End-of-video processing - checking 1 tracks...
🔄 Processing 3 pending validations...
✅ Bird #257 disappeared quickly after entry - REAL ENTRY
🐦 FINAL: Bird #257 validated and counted
✅ Bird #258 disappeared quickly after entry - REAL ENTRY
🐦 FINAL: Bird #258 validated and counted
⚪ Bird #259 has reasonable track (len=3, avg_dist=98.5) - keeping count
🐦 FINAL: Bird #259 validated and counted

📊 TIGHTENED VALIDATION SUMMARY:
   Total birds processed for validation: 120
   Validated as real entries: 113
   Rejected as false positives: 7
   False positive rejection rate: 5.8%
   🔍 Validation parameters:
     Disappear threshold: 4 frames
     Movement away threshold: 20px
     Horizontal movement ratio: 2.5


📊 Results JSON saved: drive/MyDrive/UW-MS-DS/CSE493G1/Final_Project/Github/swiftcounter-cv/experiments/data/sam2_results_downloaded_video_9-52_to_10-10_segment_5_extreme_density.json

✅ PROCESSING COMPLETE!
🐦 FINAL COUNT: 113 birds
📊 End-of-video adds: +0 birds
✅ Validated as real: 113 birds
❌ Rejected as false: 7 birds
🔍 Post-entry validation: ENABLED
📹 Output video: drive/MyDrive/UW-MS-DS/CSE493G1/Final_Project/Github/swiftcounter-cv/experiments/data/sam2_output_downloaded_video_9-52_to_10-10_segment_5_extreme_density.mp4
✅ segment_5_extreme_density: 113 birds detected
   Accuracy: 77.9% (NEEDS_WORK)

📊 COMPREHENSIVE TEST RESULTS SUMMARY
Segment            Time Range      Expected   Actual     Accuracy     Status         
------------------ --------------- ---------- ---------- ------------ ---------------
segment_1_baselin  0:05 to 0:16    47         36         76.6%        NEEDS_WORK     
segment_2_medium   0:33 to 0:54    71         64         90.1%        EXCELLENT      
segment_